Diagnostics on which local embedding version performs better - delta or as is

In [0]:
"""
KNN diagnostic
"""
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, brier_score_loss, balanced_accuracy_score
)

# ============================================================================
# Config
# ============================================================================
SOURCE_TABLES = {
    "cumulative": "catalog.schema.patient_clmbr_embeddings_new_v5_rolling_month",
    "delta":      "catalog.schema.patient_clmbr_embeddings_new_v5_rolling_trimester",
}
COHORT_TABLE = "catalog.schema.step7_filtered_cohort_with_apos"

EMBED_DIM     = 768
CLASS_NAMES   = ["control", "pe_ghtn", "gdm"]
OUTCOME_NAMES = ["pe_ghtn", "gdm"]   # matches the rest of the pipeline's convention

N_NEIGHBORS    = 15
VAL_FRACTION   = 0.2
RANDOM_SEED    = 42

MAX_TRAIN_PATIENTS = 4000 
PULL_CHUNK_SIZE     = 2000 

USE_PCA = True
PCA_VARIANCE_THRESHOLD = 0.90   # same "dims_90" convention used elsewhere in this project
PCA_MAX_COMPONENTS     = 200    # safety cap even if the variance threshold would want more


# ============================================================================
# Step 1: cohort labels
# ============================================================================
print("Loading cohort labels...")
cohort_df = spark.table(COHORT_TABLE).select(
    "person_id", "has_pe", "has_inferred_pe", "has_ghtn", "has_gdm", "is_control"
).toPandas()

cohort_df["y_control"] = cohort_df["is_control"].astype(int)
cohort_df["y_pe_ghtn"] = (
    (cohort_df["has_pe"] == 1) | (cohort_df["has_inferred_pe"] == 1) | (cohort_df["has_ghtn"] == 1)
).astype(int)
cohort_df["y_gdm"] = cohort_df["has_gdm"].astype(int)
label_cols = ["y_control", "y_pe_ghtn", "y_gdm"]
cohort_df = cohort_df[["person_id"] + label_cols].rename(columns={"person_id": "patient_id"})
cohort_df["patient_id"] = cohort_df["patient_id"].astype(str)  # match embedding tables' patient_id dtype
print(f"  {len(cohort_df)} patients with labels.")


# ============================================================================
# Step 2: full metric suite
# ============================================================================
def evaluate_multilabel(y_true, y_prob, class_names=CLASS_NAMES, outcome_names=OUTCOME_NAMES):
    results = {}
    y_pred = (y_prob >= 0.5).astype(int)

    for c, name in enumerate(class_names):
        yt, yp, ypred = y_true[:, c], y_prob[:, c], y_pred[:, c]
        n_pos, n_neg = int(yt.sum()), int((1 - yt).sum())

        results[f"auroc_{name}"] = float(roc_auc_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"auprc_{name}"] = float(average_precision_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"f1_{name}"]    = float(f1_score(yt, ypred, zero_division=0)) if n_pos > 0 else float("nan")
        results[f"brier_{name}"] = float(brier_score_loss(yt, yp))
        results[f"bacc_{name}"]  = float(balanced_accuracy_score(yt, ypred)) if n_pos > 0 and n_neg > 0 else float("nan")

    auroc_vals = [results[f"auroc_{n}"] for n in class_names if not np.isnan(results[f"auroc_{n}"])]
    results["auroc_macro"] = float(np.mean(auroc_vals)) if auroc_vals else float("nan")

    weights, wvals = [], []
    for c, name in enumerate(class_names):
        v = results[f"auroc_{name}"]
        if not np.isnan(v):
            w = float(y_true[:, c].sum()); weights.append(w); wvals.append(v * w)
    results["auroc_weighted"] = float(sum(wvals) / sum(weights)) if weights else float("nan")

    try:
        results["auroc_micro"] = float(roc_auc_score(y_true, y_prob, average="micro"))
    except Exception:
        results["auroc_micro"] = float("nan")

    out_auroc = [results[f"auroc_{n}"] for n in outcome_names if not np.isnan(results[f"auroc_{n}"])]
    results["auroc_macro_outcomes_only"] = float(np.mean(out_auroc)) if out_auroc else float("nan")

    auprc_vals = [results[f"auprc_{n}"] for n in class_names if not np.isnan(results[f"auprc_{n}"])]
    results["auprc_macro"] = float(np.mean(auprc_vals)) if auprc_vals else float("nan")

    out_auprc = [results[f"auprc_{n}"] for n in outcome_names if not np.isnan(results[f"auprc_{n}"])]
    results["auprc_macro_outcomes_only"] = float(np.mean(out_auprc)) if out_auprc else float("nan")

    return results


def print_metrics(results, prevalence, class_names=CLASS_NAMES):
    print(f"\n  {'Metric':<32} Value   Baseline")
    print(f"  {'-'*32}  -----   --------")
    for k in ([f"auroc_{n}" for n in class_names]
              + ["auroc_macro", "auroc_weighted", "auroc_micro", "auroc_macro_outcomes_only"]):
        marker = " ← AUROC primary" if k == "auroc_macro_outcomes_only" else ""
        print(f"  {k:<32} {results.get(k, float('nan')):.4f}   0.500{marker}")
    print()
    for c, name in enumerate(class_names):
        k = f"auprc_{name}"
        print(f"  {k:<32} {results.get(k, float('nan')):.4f}   {prevalence[c]:.3f}")
    for k in ["auprc_macro", "auprc_macro_outcomes_only"]:
        marker = " ← primary metric" if k == "auprc_macro_outcomes_only" else ""
        print(f"  {k:<32} {results.get(k, float('nan')):.4f}  {marker}")
    print()
    for prefix, better in [("f1", "higher"), ("brier", "lower"), ("bacc", "higher")]:
        for name in class_names:
            k = f"{prefix}_{name}"
            print(f"  {k:<32} {results.get(k, float('nan')):.4f}   ({better} is better)")
        print()


# ============================================================================
# Step 3: chunked, filtered pull 
# ============================================================================
def pull_embeddings_for_patients(table_name, patient_ids, embed_dim, chunk_size=PULL_CHUNK_SIZE):
    patient_ids = list(patient_ids)
    chunks = []
    n_chunks = (len(patient_ids) + chunk_size - 1) // chunk_size
    for i in range(0, len(patient_ids), chunk_size):
        batch_ids = patient_ids[i:i + chunk_size]
        id_list_sql = ", ".join(f"'{pid}'" for pid in batch_ids)
        query = f"""
            SELECT CAST(patient_id AS STRING) AS patient_id, *
            FROM {table_name}
            WHERE CAST(patient_id AS STRING) IN ({id_list_sql}) AND unavailable = 0
        """
        chunk_df = spark.sql(query).toPandas()
        # drop the duplicate un-cast patient_id column that '*' would otherwise re-include
        chunk_df = chunk_df.loc[:, ~chunk_df.columns.duplicated()]
        emb_cols = [f"emb_{d}" for d in range(embed_dim)]
        chunk_df[emb_cols] = chunk_df[emb_cols].astype(np.float32)
        chunks.append(chunk_df)
        print(f"    pulled chunk {i//chunk_size + 1}/{n_chunks}: {len(chunk_df)} rows")
    return pd.concat(chunks, ignore_index=True)


# ============================================================================
# Step 4: run the KNN diagnostic for one embedding table
# ============================================================================
def run_knn_diagnostic(table_name, cohort_df, label_cols, embed_dim=EMBED_DIM,
                       n_neighbors=N_NEIGHBORS, val_fraction=VAL_FRACTION,
                       max_train_patients=MAX_TRAIN_PATIENTS, seed=RANDOM_SEED,
                       use_pca=USE_PCA, pca_variance_threshold=PCA_VARIANCE_THRESHOLD,
                       pca_max_components=PCA_MAX_COMPONENTS):
    print(f"\n{'='*70}\n{table_name}\n{'='*70}")

    # ---- lightweight: only distinct patient_id, no embeddings yet ----
    print("  Getting distinct available patient_ids (no embeddings pulled yet)...")
    avail_pids = spark.sql(
        f"SELECT DISTINCT patient_id FROM {table_name} WHERE unavailable = 0"
    ).toPandas()
    avail_pids["patient_id"] = avail_pids["patient_id"].astype(str)
    print(f"  {len(avail_pids)} patients have at least one available window.")

    # ---- join with cohort labels (both small, pure pandas) ----
    patient_labels = avail_pids.merge(cohort_df, on="patient_id", how="inner")
    print(f"  {len(patient_labels)} patients after joining cohort labels.")

    # ---- patient-level split (still no embeddings touched) ----
    combo = (patient_labels["y_control"]*4 + patient_labels["y_pe_ghtn"]*2 + patient_labels["y_gdm"]).astype(int)
    train_pids, val_pids = train_test_split(
        patient_labels["patient_id"], test_size=val_fraction, stratify=combo, random_state=seed
    )

    # ---- cap TRAINING patients BEFORE pulling any embedding data ----
    if len(train_pids) > max_train_patients:
        train_combo = combo.loc[train_pids.index]
        train_pids, _ = train_test_split(
            train_pids, train_size=max_train_patients, stratify=train_combo, random_state=seed
        )
        print(f"  Training patients capped to {len(train_pids)} (MAX_TRAIN_PATIENTS={max_train_patients}).")
    print(f"  Final: {len(train_pids)} train patients, {len(val_pids)} val patients "
          f"(val NOT capped -- full validation read).")

    # ---- NOW pull embeddings, chunked, for exactly these patients ----
    print(f"  Pulling training embeddings ({len(train_pids)} patients)...")
    train_df = pull_embeddings_for_patients(table_name, train_pids, embed_dim)
    print(f"  Pulling validation embeddings ({len(val_pids)} patients)...")
    val_df = pull_embeddings_for_patients(table_name, val_pids, embed_dim)

    train_df = train_df.merge(cohort_df, on="patient_id", how="inner")
    val_df   = val_df.merge(cohort_df, on="patient_id", how="inner")
    print(f"  Row-level: {len(train_df)} train rows, {len(val_df)} val rows.")

    emb_cols = [f"emb_{d}" for d in range(embed_dim)]
    for name, d in [("train", train_df), ("val", val_df)]:
        if d[emb_cols].isna().any().any():
            n_bad = d[emb_cols].isna().any(axis=1).sum()
            print(f"  WARNING: {n_bad} {name} rows have NaN embeddings despite unavailable==0 -- dropping.")
            d.dropna(subset=emb_cols, inplace=True)

    X_train = train_df[emb_cols].values.astype(np.float32)
    Y_train = train_df[label_cols].values.astype(int)
    X_val   = val_df[emb_cols].values.astype(np.float32)
    Y_val   = val_df[label_cols].values.astype(int)
    del train_df, val_df  # release the pandas frames now that we have plain arrays

    if use_pca:
        # fit on the FINAL training rows only (post row-subsample) -- PCA
        # and KNN then see exactly the same data, and val never leaks in.
        pca_probe = PCA(random_state=seed).fit(X_train)
        cumvar = np.cumsum(pca_probe.explained_variance_ratio_)
        n_components = int(np.argmax(cumvar >= pca_variance_threshold)) + 1
        n_components = min(n_components, pca_max_components, X_train.shape[1])
        pca = PCA(n_components=n_components, random_state=seed).fit(X_train)
        X_train = pca.transform(X_train)
        X_val   = pca.transform(X_val)
        actual_var = cumvar[n_components - 1]
        print(f"  PCA: {n_components} components explain {actual_var*100:.1f}% variance "
              f"(target {pca_variance_threshold*100:.0f}%, capped at {pca_max_components}) "
              f"-- {X_train.shape[1]}-dim reduced from {embed_dim}-dim.")

    print(f"  Fitting KNN (k={n_neighbors}) on {len(X_train)} rows, {X_train.shape[1]} dims...")
    knn = KNeighborsClassifier(n_neighbors=n_neighbors, algorithm="auto", n_jobs=-1)
    knn.fit(X_train, Y_train)

    proba_list = knn.predict_proba(X_val)   # list of (n_val, 2) arrays, one per label
    Y_prob = np.column_stack([
        p[:, 1] if p.shape[1] == 2 else np.zeros(len(X_val))   # guard: label with only one class in train
        for p in proba_list
    ])

    prevalence = Y_train.mean(axis=0)
    results = evaluate_multilabel(Y_val, Y_prob)
    print(f"\n=== {table_name} — validation metrics (row-level, k={n_neighbors}) ===")
    print_metrics(results, prevalence)

    return results


# ============================================================================
# Step 5: run for both tables, compare
# ============================================================================
all_results = {}
for label, table_name in SOURCE_TABLES.items():
    all_results[label] = run_knn_diagnostic(table_name, cohort_df, label_cols)

print(f"\n{'='*70}\nCOMPARISON\n{'='*70}")
print(f"{'Metric':<30} {'cumulative':>12} {'delta':>12}")
compare_keys = ["auprc_macro_outcomes_only", "auroc_macro_outcomes_only",
                "auprc_macro", "auroc_macro"]
for k in compare_keys:
    cv, dv = all_results["cumulative"][k], all_results["delta"][k]
    marker = " ← primary" if k == "auprc_macro_outcomes_only" else ""
    print(f"{k:<30} {cv:>12.4f} {dv:>12.4f}{marker}")

primary = "auprc_macro_outcomes_only"
winner = "cumulative" if all_results["cumulative"][primary] > all_results["delta"][primary] else "delta"
print(f"\nHigher validation {primary}: {winner} "
      f"({all_results[winner][primary]:.4f} vs "
      f"{all_results['delta' if winner=='cumulative' else 'cumulative'][primary]:.4f})")

In [0]:
%pip install torch
%restart_python

In [0]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split

CFG = {
    "flags_path": "catalog.schema.step7_filtered_cohort_with_apos",
    "rolling_tables": {
        "delta":     "catalog.schema.patient_clmbr_embeddings_new_v5_rolling_trimester_delta",
        "month": "catalog.schema.patient_clmbr_embeddings_new_v5_rolling_month",
    },
    "embedding_dim": 768,
    "top_k": 5,
    "random_seed": 42,
    "max_train_patients_per_stratum": None,
}

CLASS_NAMES = ["control", "pe_ghtn", "gdm"]


def load_patient_labels(cfg):
    """Person-level labels only -- used both to build the stratified
    person-level split and to broadcast weak labels onto window rows.
    Mirrors load_and_split's label construction exactly, so results are
    comparable to your other probes."""
    flags_df = spark.table(cfg["flags_path"]).select(
        "person_id", "has_pe", "has_ghtn", "has_gdm", "is_control", "has_inferred_pe"
    ).toPandas()

    flags_df["y_control"] = flags_df["is_control"].astype(int)
    flags_df["y_pe_ghtn"] = (
        (flags_df["has_pe"] == 1) | (flags_df["has_inferred_pe"] == 1) | (flags_df["has_ghtn"] == 1)
    ).astype(int)
    flags_df["y_gdm"] = flags_df["has_gdm"].astype(int)
    return flags_df[["person_id", "y_control", "y_pe_ghtn", "y_gdm"]]


def split_patient_ids(labels_df, cfg):
    """Person-level stratified 80/10/10 split, same combo-stratification
    logic as load_and_split -- done on PATIENT ids only, before any
    window-row data is touched, so a patient's windows never straddle
    train/val/test."""
    combo = (labels_df["y_control"]*4 + labels_df["y_pe_ghtn"]*2 + labels_df["y_gdm"]).astype(int)
    pid_tv, pid_test = train_test_split(
        labels_df["person_id"].values, test_size=0.10, stratify=combo, random_state=cfg["random_seed"]
    )
    combo_tv = combo[labels_df["person_id"].isin(pid_tv)].values
    pid_train, pid_val = train_test_split(
        pid_tv, test_size=0.10/0.90, stratify=combo_tv, random_state=cfg["random_seed"]
    )
    return set(pid_train), set(pid_val), set(pid_test)


def load_rolling_windows(table_name, embedding_dim):
    """Loads a long-format rolling table (patient_id, window_name,
    emb_0..emb_{D-1}, n_tokens, unavailable, empty), dropping rows where
    unavailable=1 (window_end fell before first event or beyond the
    patient's observed data -- not a real embedding, per
    extract_windows_vectorized's semantics)."""
    df = spark.table(table_name).toPandas()
    df = df[df["unavailable"] == 0].reset_index(drop=True)
    emb_cols = sorted(
        [c for c in df.columns if c.startswith("emb_")],
        key=lambda x: int(x.split("_")[1])
    )
    assert len(emb_cols) == embedding_dim
    return df, emb_cols


def stratified_sample_patient_ids(patient_labels, combo, max_per_stratum, seed):
    rng = np.random.RandomState(seed)
    sampled = []
    for stratum in np.unique(combo):
        stratum_pids = patient_labels.loc[combo == stratum, "patient_id"].values
        if len(stratum_pids) > max_per_stratum:
            stratum_pids = rng.choice(stratum_pids, size=max_per_stratum, replace=False)
        sampled.append(stratum_pids)
    return np.concatenate(sampled)

def train_mi_svm_probe(X_train_rows, y_train_rows, patient_ids_train, top_m=1, n_em_iters=5, verbose=True):
    y_train_rows = np.asarray(y_train_rows)
    pos_mask_all = (y_train_rows == 1)
    neg_mask_all = ~pos_mask_all

    # negative rows are fixed for the whole procedure
    X_neg, y_neg = X_train_rows[neg_mask_all], y_train_rows[neg_mask_all]

    # start: broadcast (same as before) -- gives the first iteration
    # something reasonable to score positive-patient windows with
    selected_pos_idx = np.where(pos_mask_all)[0]

    prev_selected = None
    for it in range(1, n_em_iters + 1):
        X_fit = np.vstack([X_neg, X_train_rows[selected_pos_idx]])
        y_fit = np.concatenate([y_neg, y_train_rows[selected_pos_idx]])

        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_fit, y_fit)

        # re-score ALL positive-patient windows (not just currently
        # selected ones) so a previously-dropped window can be
        # reconsidered if the updated model now favors it
        pos_idx_all = np.where(pos_mask_all)[0]
        scores = clf.decision_function(X_train_rows[pos_idx_all])
        pids_pos = patient_ids_train[pos_idx_all]

        new_selected = []
        for pid in np.unique(pids_pos):
            pid_mask = (pids_pos == pid)
            pid_local_idx = pos_idx_all[pid_mask]
            pid_scores = scores[pid_mask]
            top_m_local = pid_local_idx[np.argsort(-pid_scores)[:min(top_m, len(pid_scores))]]
            new_selected.extend(top_m_local.tolist())
        new_selected = np.array(sorted(new_selected))

        if verbose:
            print(f"    MI-SVM iter {it}: {len(new_selected)} positive instances selected "
                  f"(from {len(pos_idx_all)} candidate positive-patient windows)")

        if prev_selected is not None and np.array_equal(new_selected, prev_selected):
            if verbose:
                print(f"    Converged at iter {it}.")
            selected_pos_idx = new_selected
            break
        prev_selected = new_selected
        selected_pos_idx = new_selected

    # final fit on the converged selection
    X_fit = np.vstack([X_neg, X_train_rows[selected_pos_idx]])
    y_fit = np.concatenate([y_neg, y_train_rows[selected_pos_idx]])
    final_clf = LogisticRegression(max_iter=1000)
    final_clf.fit(X_fit, y_fit)
    return final_clf


def pull_embeddings_for_patients(table_name, patient_ids, embed_dim, chunk_size=2000, verbose=True):
    """Chunked, patient-filtered pull -- ported directly from the KNN
    diagnostic script. Only ever brings the given patients' unavailable=0
    rows into pandas, in chunks, so peak driver memory is bounded by one
    chunk rather than the whole table. Casts embeddings to float32 on
    arrival (halves memory vs the float64 Spark->pandas default).

    NOTE: emb_k columns are explicitly CAST(... AS DOUBLE) in the SQL
    itself (not left to Spark's inferred column type). Delta-table emb
    columns can be all-NaN within a given partition (see
    drop_nan_embedding_rows docstring -- no-ramp-up windows), and when a
    partition's column is 100% null, Spark/Arrow can infer that
    partition's column as `null` type instead of `double`. toPandas()
    with Arrow enabled then fails with
    'ArrowInvalid: Schema at index N was different' when it tries to
    concatenate that partition's RecordBatch against another partition
    where the same column inferred as `double`. Casting in SQL pins the
    column type before Arrow ever sees it, so every partition agrees.
    """
    patient_ids = list(patient_ids)
    emb_cols = [f"emb_{d}" for d in range(embed_dim)]
    chunks = []
    n_chunks = (len(patient_ids) + chunk_size - 1) // chunk_size
    for i in range(0, len(patient_ids), chunk_size):
        batch_ids = patient_ids[i:i + chunk_size]
        id_list_sql = ", ".join(f"'{pid}'" for pid in batch_ids)
        emb_select_sql = ", ".join(f"CAST({c} AS DOUBLE) AS {c}" for c in emb_cols)
        query = f"""
            SELECT * EXCEPT ({", ".join(emb_cols)}), {emb_select_sql}
            FROM {table_name}
            WHERE patient_id IN ({id_list_sql}) AND unavailable = 0
        """
        chunk_df = spark.sql(query).toPandas()
        chunk_df[emb_cols] = chunk_df[emb_cols].astype(np.float32)
        chunk_df["patient_id"] = chunk_df["patient_id"].astype(str)
        chunks.append(chunk_df)
        if verbose:
            print(f"    pulled chunk {i//chunk_size + 1}/{n_chunks}: {len(chunk_df)} rows")
    return pd.concat(chunks, ignore_index=True)

def drop_nan_embedding_rows(df, emb_cols, label, verbose=True):
    """Drops rows with any NaN across the embedding columns. Needed
    specifically for DELTA tables: unavailable=0 only means the SOURCE
    window had data -- it doesn't guarantee the delta itself was
    computable (e.g. the no-ramp-up delta script leaves early windows
    NaN when a full-duration predecessor doesn't exist yet, even though
    unavailable=0 for that row). For non-delta (raw cumulative) tables
    this should be a no-op (0 rows dropped) -- printing the drop count
    either way so you can confirm which case you're in."""
    n_before = len(df)
    mask = df[emb_cols].isna().any(axis=1)
    n_dropped = int(mask.sum())
    if n_dropped > 0 and verbose:
        print(f"    {label}: dropped {n_dropped}/{n_before} rows with NaN embeddings "
              f"(unavailable=0 but delta not yet computable, or similar leading/trailing gap)")
    return df.loc[~mask].reset_index(drop=True)

def topk_pool_probe(table_name, cfg, verbose=True):
    """Same top-k pooling logic as before, but now memory-bounded:
    (1) pull only DISTINCT patient_id first (tiny), (2) split at the
    patient level using just those IDs, (3) cap TRAINING patients per
    combo-stratum via cfg['max_train_patients_per_stratum'] BEFORE any
    embedding data is pulled (val is NOT capped -- you want a real read
    on validation performance), (4) pull embeddings via chunked
    WHERE patient_id IN (...) queries, bounding peak memory to one
    chunk instead of the whole table.
    """
    labels_df = load_patient_labels(cfg)

    if verbose:
        print(f"\n=== {table_name} ===")
        print("  Getting distinct available patient_ids (no embeddings pulled yet)...")
    avail_pids = spark.sql(
        f"SELECT DISTINCT patient_id FROM {table_name} WHERE unavailable = 0"
    ).toPandas()
    avail_pids["patient_id"] = avail_pids["patient_id"].astype(str)
    labels_df["person_id"] = labels_df["person_id"].astype(str)

    patient_labels = avail_pids.merge(labels_df, left_on="patient_id", right_on="person_id", how="inner")
    if verbose:
        print(f"  {len(patient_labels)} patients after joining cohort labels.")

    combo = (patient_labels["y_control"]*4 + patient_labels["y_pe_ghtn"]*2 + patient_labels["y_gdm"]).astype(int).values
    train_pids_all, val_pids = train_test_split(
        patient_labels["patient_id"].values, test_size=0.10/0.90, stratify=combo, random_state=cfg["random_seed"]
    )
    # NOTE: this reproduces only the train/val portion of the 80/10/10
    # split (test set setaside not needed for this probe) -- if you need
    # test held out consistently with load_and_split elsewhere, do the
    # 3-way split first and discard test's ids, rather than this 2-way
    # split, since test_size math differs otherwise.

    max_per_stratum = cfg.get("max_train_patients_per_stratum")
    if max_per_stratum is not None:
        train_combo = combo[np.isin(patient_labels["patient_id"].values, train_pids_all)]
        train_labels_subset = patient_labels[patient_labels["patient_id"].isin(train_pids_all)]
        train_pids = stratified_sample_patient_ids(train_labels_subset, train_combo, max_per_stratum, cfg["random_seed"])
        if verbose:
            print(f"  Training patients capped to {len(train_pids)} "
                  f"(max_train_patients_per_stratum={max_per_stratum}, from {len(train_pids_all)} available).")
    else:
        train_pids = train_pids_all

    if verbose:
        print(f"  Final: {len(train_pids)} train patients, {len(val_pids)} val patients "
              f"(val NOT capped -- full validation read).")
        print(f"  Pulling training embeddings ({len(train_pids)} patients)...")
    train_df, emb_cols = pull_embeddings_for_patients(table_name, train_pids, cfg["embedding_dim"]), None
    emb_cols = sorted([c for c in train_df.columns if c.startswith("emb_")], key=lambda x: int(x.split("_")[1]))
    assert len(emb_cols) == cfg["embedding_dim"]

    if verbose:
        print(f"  Pulling training embeddings ({len(train_pids)} patients)...")
    train_df, emb_cols = pull_embeddings_for_patients(table_name, train_pids, cfg["embedding_dim"]), None
    emb_cols = sorted([c for c in train_df.columns if c.startswith("emb_")], key=lambda x: int(x.split("_")[1]))
    assert len(emb_cols) == cfg["embedding_dim"]
    train_df = drop_nan_embedding_rows(train_df, emb_cols, "train", verbose=verbose)

    if verbose:
        print(f"  Pulling validation embeddings ({len(val_pids)} patients)...")
    val_df = pull_embeddings_for_patients(table_name, val_pids, cfg["embedding_dim"])
    val_df = drop_nan_embedding_rows(val_df, emb_cols, "val", verbose=verbose)

    train_rows = train_df.merge(labels_df, left_on="patient_id", right_on="person_id", how="inner")
    val_rows   = val_df.merge(labels_df, left_on="patient_id", right_on="person_id", how="inner")
    del train_df, val_df

    if verbose:
        print(f"  train: {train_rows['patient_id'].nunique()} patients / {len(train_rows)} window-rows")
        print(f"  val:   {val_rows['patient_id'].nunique()} patients / {len(val_rows)} window-rows")

    results = {}
    for class_idx, name in enumerate(CLASS_NAMES):
        label_col = f"y_{name}"
        X_train_rows = train_rows[emb_cols].values.astype(np.float32)
        y_train_rows = train_rows[label_col].values.astype(int)

        # clf = LogisticRegression(max_iter=1000)
        # clf.fit(X_train_rows, y_train_rows)
        clf = train_mi_svm_probe(
            X_train_rows, y_train_rows,
            train_rows["patient_id"].values,
            top_m=1, n_em_iters=5, verbose=verbose,
        )

        X_val_rows = val_rows[emb_cols].values.astype(np.float32)
        y_val_rows = val_rows[label_col].values.astype(int)
        p_val_rows = clf.predict_proba(X_val_rows)[:, 1]
        window_level_auroc = roc_auc_score(y_val_rows, p_val_rows) if len(np.unique(y_val_rows)) > 1 else float("nan")

        val_rows_scored = val_rows[["patient_id", label_col]].copy()
        val_rows_scored["p"] = p_val_rows

        pooled_scores, pooled_labels = [], []
        for pid, group in val_rows_scored.groupby("patient_id"):
            top_probs = group["p"].nlargest(min(cfg["top_k"], len(group))).values
            pooled_scores.append(top_probs.mean())
            pooled_labels.append(group[label_col].iloc[0])
        pooled_scores = np.array(pooled_scores)
        pooled_labels = np.array(pooled_labels)

        pooled_auroc = roc_auc_score(pooled_labels, pooled_scores) if len(np.unique(pooled_labels)) > 1 else float("nan")
        pooled_brier = brier_score_loss(pooled_labels, pooled_scores)

        results[name] = {
            "window_level_auroc": window_level_auroc,
            "pooled_auroc": pooled_auroc,
            "pooled_brier": pooled_brier,
            "n_val_patients": len(pooled_labels),
        }
        if verbose:
            print(f"  {name:10s}: window-level AUROC {window_level_auroc:.4f}  |  "
                  f"top-{cfg['top_k']}-pooled AUROC {pooled_auroc:.4f}  Brier {pooled_brier:.4f}  "
                  f"(n_val_patients={len(pooled_labels)})")

    return results


if __name__ == "__main__":
    all_results = {}
    for granularity, table_name in CFG["rolling_tables"].items():
        all_results[granularity] = topk_pool_probe(table_name, CFG)

    print("\n=== Summary: top-k pooled AUROC by granularity ===")
    for granularity, res in all_results.items():
        for name in CLASS_NAMES:
            print(f"  {granularity:10s} / {name:10s}: {res[name]['pooled_auroc']:.4f}")

In [0]:
import gc
gc.collect()

In [0]:
def fast_topk_pool_probe(table_name, cfg, n_pca_components=100, verbose=True):
    """Fast A/B-comparison version of topk_pool_probe: ONE plain
    LogisticRegression fit per class (no MI-SVM iteration), PCA-reduced
    embeddings, and a capped training set. This trades away MI-SVM's
    label-denoising benefit in exchange for speed -- appropriate here
    because the goal is comparing delta vs non-delta RELATIVELY, and
    the label-noise simplification affects both datasets equally, so it
    shouldn't bias the comparison between them. NOT intended as a final
    model -- once you've picked a dataset, go back to the full MI-SVM
    procedure on that one dataset only.
    """
    from sklearn.decomposition import PCA

    labels_df = load_patient_labels(cfg)

    if verbose:
        print(f"\n=== {table_name} (fast comparison mode) ===")
    avail_pids = spark.sql(
        f"SELECT DISTINCT patient_id FROM {table_name} WHERE unavailable = 0"
    ).toPandas()
    avail_pids["patient_id"] = avail_pids["patient_id"].astype(str)
    labels_df["person_id"] = labels_df["person_id"].astype(str)
    patient_labels = avail_pids.merge(labels_df, left_on="patient_id", right_on="person_id", how="inner")

    combo = (patient_labels["y_control"]*4 + patient_labels["y_pe_ghtn"]*2 + patient_labels["y_gdm"]).astype(int).values
    train_pids_all, val_pids = train_test_split(
        patient_labels["patient_id"].values, test_size=0.10/0.90, stratify=combo, random_state=cfg["random_seed"]
    )

    max_per_stratum = cfg.get("max_train_patients_per_stratum") or 500  # force a cap for speed here regardless of CFG
    train_combo = combo[np.isin(patient_labels["patient_id"].values, train_pids_all)]
    train_labels_subset = patient_labels[patient_labels["patient_id"].isin(train_pids_all)]
    train_pids = stratified_sample_patient_ids(train_labels_subset, train_combo, max_per_stratum, cfg["random_seed"])
    if verbose:
        print(f"  Training patients capped to {len(train_pids)} (from {len(train_pids_all)} available)")

    # cap validation too, for THIS fast comparison only -- full val read
    # is what made the earlier run slow at scoring time as well as fit
    # time; a few thousand val patients is enough to compare two datasets
    max_val_patients = cfg.get("fast_compare_max_val_patients", 3000)
    if len(val_pids) > max_val_patients:
        val_combo = combo[np.isin(patient_labels["patient_id"].values, val_pids)]
        val_labels_subset = patient_labels[patient_labels["patient_id"].isin(val_pids)]
        val_pids = stratified_sample_patient_ids(val_labels_subset, val_combo,
                                                  max_val_patients // len(np.unique(combo)) + 1, cfg["random_seed"])
        if verbose:
            print(f"  Val patients capped to {len(val_pids)} for fast comparison")

    train_df = pull_embeddings_for_patients(table_name, train_pids, cfg["embedding_dim"])
    emb_cols = sorted([c for c in train_df.columns if c.startswith("emb_")], key=lambda x: int(x.split("_")[1]))
    train_df = drop_nan_embedding_rows(train_df, emb_cols, "train", verbose=verbose)
    val_df = pull_embeddings_for_patients(table_name, val_pids, cfg["embedding_dim"])
    val_df = drop_nan_embedding_rows(val_df, emb_cols, "val", verbose=verbose)

    train_rows = train_df.merge(labels_df, left_on="patient_id", right_on="person_id", how="inner")
    val_rows   = val_df.merge(labels_df, left_on="patient_id", right_on="person_id", how="inner")
    del train_df, val_df

    if verbose:
        print(f"  train: {train_rows['patient_id'].nunique()} patients / {len(train_rows)} window-rows")
        print(f"  val:   {val_rows['patient_id'].nunique()} patients / {len(val_rows)} window-rows")

    X_train_full = train_rows[emb_cols].values.astype(np.float32)
    X_val_full = val_rows[emb_cols].values.astype(np.float32)

    pca = PCA(n_components=n_pca_components, random_state=cfg["random_seed"]).fit(X_train_full)
    X_train_pca = pca.transform(X_train_full)
    X_val_pca = pca.transform(X_val_full)
    if verbose:
        print(f"  PCA: {n_pca_components} components explain "
              f"{pca.explained_variance_ratio_.sum()*100:.1f}% variance "
              f"({X_train_full.shape[1]}-dim -> {n_pca_components}-dim)")

    results = {}
    for class_idx, name in enumerate(CLASS_NAMES):
        label_col = f"y_{name}"
        y_train_rows = train_rows[label_col].values.astype(int)
        y_val_rows = val_rows[label_col].values.astype(int)

        clf = LogisticRegression(max_iter=500, class_weight="balanced")
        clf.fit(X_train_pca, y_train_rows)

        p_val_rows = clf.predict_proba(X_val_pca)[:, 1]
        window_level_auroc = roc_auc_score(y_val_rows, p_val_rows) if len(np.unique(y_val_rows)) > 1 else float("nan")

        val_rows_scored = val_rows[["patient_id", label_col]].copy()
        val_rows_scored["p"] = p_val_rows
        pooled_scores, pooled_labels = [], []
        for pid, group in val_rows_scored.groupby("patient_id"):
            top_probs = group["p"].nlargest(min(cfg["top_k"], len(group))).values
            pooled_scores.append(top_probs.mean())
            pooled_labels.append(group[label_col].iloc[0])
        pooled_scores, pooled_labels = np.array(pooled_scores), np.array(pooled_labels)

        pooled_auroc = roc_auc_score(pooled_labels, pooled_scores) if len(np.unique(pooled_labels)) > 1 else float("nan")
        pooled_brier = brier_score_loss(pooled_labels, pooled_scores)

        results[name] = {"window_level_auroc": window_level_auroc, "pooled_auroc": pooled_auroc,
                         "pooled_brier": pooled_brier, "n_val_patients": len(pooled_labels)}
        if verbose:
            print(f"  {name:10s}: window-level AUROC {window_level_auroc:.4f}  |  "
                  f"top-{cfg['top_k']}-pooled AUROC {pooled_auroc:.4f}  Brier {pooled_brier:.4f}")

    return results

In [0]:
all_results = {}
for granularity, table_name in CFG["rolling_tables"].items():
    all_results[granularity] = fast_topk_pool_probe(table_name, CFG, n_pca_components=100)

In [0]:
import os, time, copy, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, brier_score_loss, balanced_accuracy_score
)
import matplotlib.pyplot as plt

# ============================================================================
# Section 1: Config (as given)
# ============================================================================
CLASS_NAMES   = ["control", "pe_ghtn", "gdm"]
OUTCOME_NAMES = ["pe_ghtn", "gdm"]

DEFAULT_CFG = {
    "embeddings_path":  "catalog.schema.patient_clmbr_embeddings_new_v5",
    "flags_path":       "catalog.schema.step7_filtered_cohort_with_apos",
    "output_dir":       "./proto_runs",

    "embedding_dim":    768,
    "adapter_hidden":   256,
    "adapter_out":      64,

    "prototypes_per_class": {"control": 5, "pe_ghtn": 5, "gdm": 5},
    "n_classes": 3,

    "lr":              0.003,
    "weight_decay":    1e-4,
    "batch_size":      512,
    "max_epochs":      200,
    "patience":        10,

    "lambda_clst":     0.0004,
    "lambda_sep":      0.0004,
    "lambda_div":      1.0,
    "lambda_cntrst":   300.0,
    "zero_diagonal_blocks": True,

    "use_jscl_pretrain":    False,
    "pretrain_epochs":      50,
    "pretrain_lr":          1e-3,
    "pretrain_temperature": 0.2,
    "pretrain_patience":    10,
    "pretrain_batch_size":  512,

    "dims_check_interval": 5,

    "max_iterations":  5,
    "iter_patience":   2,
    "lr_decay":        0.5,

    "random_seed":     42,
    "run_id": "run_default",

    "lambda_covar": 0.5,

    # top-k pooling at inference (see module docstring)
    "topk_pool_k": 5,
}

os.makedirs(DEFAULT_CFG["output_dir"], exist_ok=True)


def total_prototypes(cfg):
    return sum(cfg["prototypes_per_class"][c] for c in CLASS_NAMES)


# ============================================================================
# Section 2: This-run config for the two monthly embedding tables
# ============================================================================
MONTH_TABLES = {
    # "cumulative": "catalog.schema.patient_clmbr_embeddings_new_v5_rolling_month",
    "delta":      "catalog.schema.patient_clmbr_embeddings_new_v5_rolling_trimester_delta",
}
COHORT_TABLE = "catalog.schema.step7_filtered_cohort_with_apos"

VAL_FRACTION       = 0.2
RANDOM_SEED         = DEFAULT_CFG["random_seed"]
MAX_TRAIN_PATIENTS  = 10000   # None = no cap (full NN training scales with epochs, not O(n^2)
                             # like KNN did -- but set a cap here if runtime needs bounding)
PULL_CHUNK_SIZE     = 2000   # patients per Spark pull -- bounds peak memory per chunk, same
                             # OOM-avoidance pattern as the KNN diagnostic script


# ============================================================================
# Section 3: Model definitions (adapter w/ BatchNorm affine=False, cosine
# PrototypeLayer -- as established throughout this project)
# ============================================================================
def class_offsets(prototypes_per_class):
    offsets = {}
    start = 0
    for name in CLASS_NAMES:
        k = prototypes_per_class[name]
        offsets[name] = (start, start + k)
        start += k
    return offsets, start


class AdapterMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
            nn.BatchNorm1d(out_dim, affine=False),
        )
    def forward(self, x):
        return self.net(x)


class PrototypeLayer(nn.Module):
    def __init__(self, prototypes_per_class, prototype_dim, scale, verbose=True):
        super().__init__()
        self.prototypes_per_class = prototypes_per_class
        self.offsets, self.n_prototypes = class_offsets(prototypes_per_class)
        self.prototype_dim = prototype_dim
        self.scale = scale
        self.prototypes = nn.Parameter(torch.randn(self.n_prototypes, prototype_dim) * 0.1)

        n_classes = len(CLASS_NAMES)
        assignment = torch.zeros(n_classes, self.n_prototypes)
        for c, name in enumerate(CLASS_NAMES):
            s, e = self.offsets[name]
            assignment[c, s:e] = 1.0
        self.register_buffer("assignment", assignment)

        if verbose:
            print(f"\nPrototype assignment ({self.n_prototypes} total, variable per class):")
            for name in CLASS_NAMES:
                s, e = self.offsets[name]
                print(f"  {name:10s}: indices {s}..{e-1}  (k={e-s})")

    def forward(self, z):
        z_norm = F.normalize(z, dim=1)
        p_norm = F.normalize(self.prototypes, dim=1)
        return self.scale * (z_norm @ p_norm.T)


class ProtoEHRNet(nn.Module):
    def __init__(self, cfg, scale, verbose=True):
        super().__init__()
        C = cfg["n_classes"]
        self.adapter = AdapterMLP(cfg["embedding_dim"], cfg["adapter_hidden"], cfg["adapter_out"])
        self.proto_layer = PrototypeLayer(cfg["prototypes_per_class"], cfg["adapter_out"], scale, verbose)
        self.classifier = nn.Linear(self.proto_layer.n_prototypes, C, bias=True)

    def forward(self, x):
        z      = self.adapter(x)
        sim    = self.proto_layer(z)
        logits = self.classifier(sim)
        return logits, sim, z


# ============================================================================
# Section 4: Loss functions
# ============================================================================
def bce_loss(logits, labels, class_weights):
    return F.binary_cross_entropy_with_logits(logits, labels, pos_weight=class_weights, reduction="mean")

def clustering_loss(sim, labels, assignment):
    pos_mask   = (labels @ assignment).clamp(0, 1).bool()
    masked_sim = sim.masked_fill(~pos_mask, -1e9)
    has_pos    = pos_mask.any(dim=1)
    if not has_pos.any():
        return sim.new_tensor(0.0)
    return -masked_sim[has_pos].max(dim=1).values.mean()

def separation_loss(sim, labels, assignment):
    neg_mask   = ((labels @ assignment) == 0)
    masked_sim = sim.masked_fill(~neg_mask, -1e9)
    has_neg    = neg_mask.any(dim=1)
    if not has_neg.any():
        return sim.new_tensor(0.0)
    return masked_sim[has_neg].max(dim=1).values.mean()

def orthogonality_loss(prototypes):
    p_norm = F.normalize(prototypes, dim=1)
    gram   = p_norm @ p_norm.T
    I      = torch.eye(gram.size(0), device=gram.device)
    return torch.norm(gram - I, p="fro") ** 2

def build_jaccard_matrix(Y_train, prototypes_per_class, zero_diagonal_blocks=True):
    n_classes = len(CLASS_NAMES)
    C_class = np.zeros((n_classes, n_classes))
    for c1 in range(n_classes):
        for c2 in range(n_classes):
            inter = ((Y_train[:, c1] == 1) & (Y_train[:, c2] == 1)).sum()
            union = ((Y_train[:, c1] == 1) | (Y_train[:, c2] == 1)).sum()
            C_class[c1, c2] = inter / union if union > 0 else 0.0
    if zero_diagonal_blocks:
        np.fill_diagonal(C_class, 0.0)
    offsets, P = class_offsets(prototypes_per_class)
    C_proto = np.zeros((P, P))
    for c1, n1 in enumerate(CLASS_NAMES):
        for c2, n2 in enumerate(CLASS_NAMES):
            rs, re = offsets[n1]; cs, ce = offsets[n2]
            C_proto[rs:re, cs:ce] = C_class[c1, c2]
    return torch.tensor(C_proto, dtype=torch.float32)

def contrastive_loss(prototypes, C):
    p_norm   = F.normalize(prototypes, dim=1)
    S        = p_norm @ p_norm.T
    C_comp   = 1.0 - C
    pos_term = (C * S).sum()      / C.sum().clamp(min=1e-8)
    neg_term = (C_comp * S).sum() / C_comp.sum().clamp(min=1e-8)
    return -(1.0 / prototypes.size(0) ** 0.5) * (pos_term - neg_term)

def jscl_loss(z, labels, temperature=0.07):
    z_norm  = F.normalize(z, dim=1)
    sim_mat = (z_norm @ z_norm.T) / temperature
    inter     = labels @ labels.T
    row_sums  = labels.sum(dim=1, keepdim=True)
    union     = row_sums + row_sums.T - inter
    jaccard   = inter / union.clamp(min=1e-8)
    self_mask = torch.eye(z.size(0), device=z.device).bool()
    jaccard   = jaccard.masked_fill(self_mask, 0)
    sim_stab = sim_mat - sim_mat.max(dim=1, keepdim=True).values
    exp_sim  = torch.exp(sim_stab).masked_fill(self_mask, 0)
    denom    = exp_sim.sum(dim=1, keepdim=True).clamp(min=1e-8)
    log_prob = sim_stab - torch.log(denom)
    weight_sum = jaccard.sum(dim=1).clamp(min=1e-8)
    loss = -(jaccard * log_prob).sum(dim=1) / weight_sum
    has_pos = (jaccard.sum(dim=1) > 0)
    if not has_pos.any():
        return z.new_tensor(0.0)
    return loss[has_pos].mean()

def covariance_loss(z, eps=1e-4):
    z = z - z.mean(dim=0, keepdim=True)
    n, d = z.shape
    cov = (z.T @ z) / (n - 1)
    off_diag = cov - torch.diag(torch.diagonal(cov))
    return (off_diag ** 2).sum() / d


# ============================================================================
# Section 5: Metrics -- full suite + top-k patient-level pooled inference
# ============================================================================
def _compute_metric_suite(y_true, y_prob, class_names=CLASS_NAMES, outcome_names=OUTCOME_NAMES):
    results = {}
    y_pred = (y_prob >= 0.5).astype(int)
    for c, name in enumerate(class_names):
        yt, yp, ypred = y_true[:, c], y_prob[:, c], y_pred[:, c]
        n_pos, n_neg = int(yt.sum()), int((1 - yt).sum())
        results[f"auroc_{name}"] = float(roc_auc_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"auprc_{name}"] = float(average_precision_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"f1_{name}"]    = float(f1_score(yt, ypred, zero_division=0)) if n_pos > 0 else float("nan")
        results[f"brier_{name}"] = float(brier_score_loss(yt, yp))
        results[f"bacc_{name}"]  = float(balanced_accuracy_score(yt, ypred)) if n_pos > 0 and n_neg > 0 else float("nan")

    auroc_vals = [results[f"auroc_{n}"] for n in class_names if not np.isnan(results[f"auroc_{n}"])]
    results["auroc_macro"] = float(np.mean(auroc_vals)) if auroc_vals else float("nan")
    weights, wvals = [], []
    for c, name in enumerate(class_names):
        v = results[f"auroc_{name}"]
        if not np.isnan(v):
            w = float(y_true[:, c].sum()); weights.append(w); wvals.append(v * w)
    results["auroc_weighted"] = float(sum(wvals) / sum(weights)) if weights else float("nan")
    try:
        results["auroc_micro"] = float(roc_auc_score(y_true, y_prob, average="micro"))
    except Exception:
        results["auroc_micro"] = float("nan")
    out_auroc = [results[f"auroc_{n}"] for n in outcome_names if not np.isnan(results[f"auroc_{n}"])]
    results["auroc_macro_outcomes_only"] = float(np.mean(out_auroc)) if out_auroc else float("nan")

    auprc_vals = [results[f"auprc_{n}"] for n in class_names if not np.isnan(results[f"auprc_{n}"])]
    results["auprc_macro"] = float(np.mean(auprc_vals)) if auprc_vals else float("nan")
    out_auprc = [results[f"auprc_{n}"] for n in outcome_names if not np.isnan(results[f"auprc_{n}"])]
    results["auprc_macro_outcomes_only"] = float(np.mean(out_auprc)) if out_auprc else float("nan")

    results["_primary"] = results["auroc_macro"]
    return results


@torch.no_grad()
def topk_pool_patient_logits(model, X, pid, device, k=5, batch_size=512):
    """Row-level forward pass (adapter + prototype layer only, no
    classifier yet), then aggregates each patient's per-window
    prototype similarities via top-k pooling -- for each prototype,
    average that patient's k highest per-window similarity scores (or
    all of them if the patient has fewer than k available windows).
    The pooled (n_patients, n_prototypes) similarity matrix is then
    passed through the classifier ONCE per patient. Verified correct in
    isolation (hand-computed cases + fewer-windows-than-k edge case)."""
    model.eval()
    all_sim = []
    for start in range(0, len(X), batch_size):
        xb = torch.tensor(X[start:start + batch_size], dtype=torch.float32).to(device)
        z = model.adapter(xb)
        sim = model.proto_layer(z)
        all_sim.append(sim.cpu().numpy())
    sim_all = np.concatenate(all_sim, axis=0)
    n_prototypes = sim_all.shape[1]

    df = pd.DataFrame(sim_all, columns=[f"proto_{i}" for i in range(n_prototypes)])
    df["patient_id"] = np.asarray(pid)

    proto_cols = [f"proto_{i}" for i in range(n_prototypes)]
    def pool_group(g):
        vals = g[proto_cols].values
        k_eff = min(k, vals.shape[0])
        topk = np.sort(vals, axis=0)[-k_eff:, :]
        return pd.Series(topk.mean(axis=0), index=proto_cols)

    pooled_df = df.groupby("patient_id", sort=False).apply(pool_group)
    pooled_pids = pooled_df.index.values
    pooled_sim = pooled_df.values.astype(np.float32)

    logits = model.classifier(torch.tensor(pooled_sim, dtype=torch.float32).to(device))
    probs = torch.sigmoid(logits).cpu().numpy()
    return pooled_pids, probs


def evaluate_patient_level(model, X, Y, pid, device, k=5,
                           class_names=CLASS_NAMES, outcome_names=OUTCOME_NAMES,
                           return_probs=False):
    """Patient-level evaluation via top-k pooled inference -- replaces
    row-level evaluate_all for all early-stopping decisions and final
    reported metrics in this script."""
    pooled_pids, probs = topk_pool_patient_logits(model, X, pid, device, k=k)

    y_cols = [f"y{i}" for i in range(Y.shape[1])]
    y_df = pd.DataFrame(Y, columns=y_cols)
    y_df["patient_id"] = np.asarray(pid)
    y_patient_df = y_df.drop_duplicates("patient_id").set_index("patient_id")
    y_patient = y_patient_df.loc[pooled_pids, y_cols].values.astype(np.float32)

    results = _compute_metric_suite(y_patient, probs, class_names, outcome_names)
    if return_probs:
        return results, probs, pooled_pids
    return results


def prototype_diagnostics(model, projection_map):
    offsets, _ = class_offsets(model.proto_layer.prototypes_per_class)
    with torch.no_grad():
        p = F.normalize(model.proto_layer.prototypes.cpu(), dim=1)
        gram = (p @ p.T).numpy()
    concentrations = []
    for name in CLASS_NAMES:
        s, e = offsets[name]; k = e - s
        if k <= 1:
            continue
        block = gram[s:e, s:e]
        off = block.sum() - np.trace(block)
        cnt = k * (k - 1)
        concentrations.append(off / cnt)
    concentration = float(np.mean(concentrations)) if concentrations else float("nan")
    sim_scores = [info["sim_score"] for info in projection_map.values()]
    fidelity_mean = float(np.mean(sim_scores)) if sim_scores else float("nan")
    fidelity_min  = float(np.min(sim_scores)) if sim_scores else float("nan")
    return {"proto_concentration": concentration, "proto_fidelity_mean": fidelity_mean,
           "proto_fidelity_min": fidelity_min}


def check_dimension_collapse(model, X_train, Y_train, device, class_names=None):
    from sklearn.decomposition import PCA
    class_names = class_names or CLASS_NAMES
    model.eval()
    all_z = []
    with torch.no_grad():
        for start in range(0, len(X_train), 512):
            xb = torch.tensor(X_train[start:start + 512], dtype=torch.float32).to(device)
            all_z.append(model.adapter(xb).cpu())
    z = torch.cat(all_z, dim=0).numpy()
    z_norm = F.normalize(torch.tensor(z), dim=1).numpy()
    pca = PCA().fit(z)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    dims_90 = int(np.argmax(cumvar >= 0.90)) + 1
    dims_95 = int(np.argmax(cumvar >= 0.95)) + 1
    # O(n*d) within-class similarity via sum-of-vectors identity (not O(n^2) gram matrix)
    within_class_sim = {}
    for c, name in enumerate(class_names):
        mask = (Y_train[:, c] == 1)
        zc = z_norm[mask]; n = len(zc)
        if n > 1:
            S = zc.sum(axis=0)
            within_class_sim[name] = (float((S * S).sum()) - n) / (n * (n - 1))
        else:
            within_class_sim[name] = float("nan")
    return {"adapter_out_dim": z.shape[1], "dims_90": dims_90, "dims_95": dims_95,
           "within_class_cosine_sim": within_class_sim}


# ============================================================================
# Section 6: Data loading -- OOM-safe chunked, filtered pull (same
# pattern as the KNN diagnostic script), row-level expansion
# ============================================================================
def to_loader(X, Y, batch_size, shuffle, device, drop_last=False):
    device = torch.device(device) if isinstance(device, str) else device
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(Y, dtype=torch.float32))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last,
                      pin_memory=(device.type == "cuda"), num_workers=0)


def pull_embeddings_for_patients(table_name, patient_ids, embed_dim, chunk_size=PULL_CHUNK_SIZE):
    patient_ids = list(patient_ids)
    if not patient_ids:
        return pd.DataFrame()
    chunks = []
    n_chunks = (len(patient_ids) + chunk_size - 1) // chunk_size
    for i in range(0, len(patient_ids), chunk_size):
        batch_ids = patient_ids[i:i + chunk_size]
        id_list_sql = ", ".join(f"'{pid}'" for pid in batch_ids)
        query = f"SELECT * FROM {table_name} WHERE patient_id IN ({id_list_sql}) AND unavailable = 0"
        chunk_df = spark.sql(query).toPandas()
        emb_cols = [f"emb_{d}" for d in range(embed_dim)]
        chunk_df[emb_cols] = chunk_df[emb_cols].astype(np.float32)
        chunk_df["patient_id"] = chunk_df["patient_id"].astype(str)
        chunks.append(chunk_df)
        print(f"    pulled chunk {i//chunk_size + 1}/{n_chunks}: {len(chunk_df)} rows")
    return pd.concat(chunks, ignore_index=True)


def load_cohort_labels():
    print("Loading cohort labels...")
    cohort_df = spark.table(COHORT_TABLE).select(
        "person_id", "has_pe", "has_inferred_pe", "has_ghtn", "has_gdm", "is_control"
    ).toPandas()
    cohort_df["y_control"] = cohort_df["is_control"].astype(int)
    cohort_df["y_pe_ghtn"] = (
        (cohort_df["has_pe"] == 1) | (cohort_df["has_inferred_pe"] == 1) | (cohort_df["has_ghtn"] == 1)
    ).astype(int)
    cohort_df["y_gdm"] = cohort_df["has_gdm"].astype(int)
    # subtype tags too, in case use_jscl_pretrain is flipped on later
    cohort_df["y_sub_control"] = cohort_df["is_control"].astype(int)
    cohort_df["y_sub_pe"]      = ((cohort_df["has_pe"] == 1) | (cohort_df["has_inferred_pe"] == 1)).astype(int)
    cohort_df["y_sub_ghtn"]    = cohort_df["has_ghtn"].astype(int)
    cohort_df["y_sub_gdm"]     = cohort_df["has_gdm"].astype(int)

    label_cols = ["y_control", "y_pe_ghtn", "y_gdm"]
    subtype_cols = ["y_sub_control", "y_sub_pe", "y_sub_ghtn", "y_sub_gdm"]
    cohort_df = cohort_df[["person_id"] + label_cols + subtype_cols].rename(columns={"person_id": "patient_id"})
    cohort_df["patient_id"] = cohort_df["patient_id"].astype(str)
    print(f"  {len(cohort_df)} patients with labels.")
    return cohort_df, label_cols, subtype_cols


def build_month_data(table_name, cohort_df, label_cols, subtype_cols, cfg,
                     val_fraction=VAL_FRACTION, seed=RANDOM_SEED,
                     max_train_patients=MAX_TRAIN_PATIENTS):
    print(f"\n{'='*70}\n{table_name}\n{'='*70}")
    embed_dim = cfg["embedding_dim"]

    print("  Getting distinct available patient_ids...")
    avail_pids = spark.sql(f"SELECT DISTINCT patient_id FROM {table_name} WHERE unavailable = 0").toPandas()
    avail_pids["patient_id"] = avail_pids["patient_id"].astype(str)

    patient_labels = avail_pids.merge(cohort_df, on="patient_id", how="inner")
    print(f"  {len(patient_labels)} patients after joining cohort labels.")

    combo = (patient_labels["y_control"]*4 + patient_labels["y_pe_ghtn"]*2 + patient_labels["y_gdm"]).astype(int)
    train_pids, val_pids = train_test_split(
        patient_labels["patient_id"], test_size=val_fraction, stratify=combo, random_state=seed
    )
    if max_train_patients is not None and len(train_pids) > max_train_patients:
        train_combo = combo.loc[train_pids.index]
        train_pids, _ = train_test_split(
            train_pids, train_size=max_train_patients, stratify=train_combo, random_state=seed
        )
        print(f"  Training patients capped to {len(train_pids)}.")
    print(f"  Final: {len(train_pids)} train patients, {len(val_pids)} val patients.")

    print(f"  Pulling training embeddings...")
    train_df = pull_embeddings_for_patients(table_name, train_pids, embed_dim)
    print(f"  Pulling validation embeddings...")
    val_df = pull_embeddings_for_patients(table_name, val_pids, embed_dim)

    all_label_cols = label_cols + subtype_cols
    train_df = train_df.merge(cohort_df[["patient_id"] + all_label_cols], on="patient_id", how="inner")
    val_df   = val_df.merge(cohort_df[["patient_id"] + all_label_cols], on="patient_id", how="inner")
    print(f"  Row-level: {len(train_df)} train rows, {len(val_df)} val rows.")

    emb_cols = [f"emb_{d}" for d in range(embed_dim)]
    for name, d in [("train", train_df), ("val", val_df)]:
        if d[emb_cols].isna().any().any():
            n_bad = d[emb_cols].isna().any(axis=1).sum()
            print(f"  WARNING: {n_bad} {name} rows have NaN embeddings -- dropping.")
            d.dropna(subset=emb_cols, inplace=True)

    X_train = train_df[emb_cols].values.astype(np.float32)
    Y_train = train_df[label_cols].values.astype(np.float32)
    pid_train = train_df["patient_id"].values
    Ysub_train = train_df[subtype_cols].values.astype(np.float32)

    X_val = val_df[emb_cols].values.astype(np.float32)
    Y_val = val_df[label_cols].values.astype(np.float32)
    pid_val = val_df["patient_id"].values
    Ysub_val = val_df[subtype_cols].values.astype(np.float32)

    pos_freq = Y_train.mean(axis=0).clip(0.03, 0.97)
    class_weights = torch.tensor((1 - pos_freq) / pos_freq, dtype=torch.float32)
    prevalence = Y_train.mean(axis=0)

    print("Label distribution (train rows):")
    for i, name in enumerate(CLASS_NAMES):
        print(f"  {name:10s}: {int(Y_train[:,i].sum()):6d} pos / {len(Y_train)} ({Y_train[:,i].mean()*100:.1f}%)")

    return {
        "X_train": X_train, "Y_train": Y_train, "pid_train": pid_train, "Y_subtype_train": Ysub_train,
        "X_val": X_val, "Y_val": Y_val, "pid_val": pid_val, "Y_subtype_val": Ysub_val,
        "class_weights": class_weights, "prevalence": prevalence,
    }


# ============================================================================
# Section 7: Training loop (Stage 0 JSCL [off by default] -> iterative
# Stage 1<->2 -> Stage 3 refit), using top-k pooled patient-level
# evaluation for all early stopping / final metrics
# ============================================================================
def pretrain_adapter_jscl(model, X_train, Y_subtype_train, X_val, Y_subtype_val, cfg, device, verbose=True):
    pretrain_bs = cfg.get("pretrain_batch_size", cfg["batch_size"])
    train_loader = to_loader(X_train, Y_subtype_train, pretrain_bs, True, device, drop_last=True)
    val_loader   = to_loader(X_val, Y_subtype_val, pretrain_bs, False, device, drop_last=False)
    for p in model.proto_layer.parameters(): p.requires_grad_(False)
    for p in model.classifier.parameters(): p.requires_grad_(False)
    opt = torch.optim.Adam(model.adapter.parameters(), lr=cfg["pretrain_lr"], weight_decay=cfg["weight_decay"])
    best_val_loss, best_state, patience_ctr = float("inf"), None, 0
    if verbose:
        print(f"\nStage 0: JSCL subtype pretraining (epochs={cfg['pretrain_epochs']}, "
              f"lr={cfg['pretrain_lr']}, temp={cfg['pretrain_temperature']}, batch_size={pretrain_bs})")
    for epoch in range(1, cfg["pretrain_epochs"] + 1):
        model.train()
        tr_losses = []
        for X_b, Y_b in train_loader:
            X_b = X_b.to(device, non_blocking=True); Y_b = Y_b.to(device, non_blocking=True)
            z = model.adapter(X_b)
            loss = jscl_loss(z, Y_b, cfg["pretrain_temperature"])
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.adapter.parameters(), 1.0)
            opt.step()
            tr_losses.append(loss.item())
        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_b, Y_b in val_loader:
                X_b = X_b.to(device, non_blocking=True); Y_b = Y_b.to(device, non_blocking=True)
                z = model.adapter(X_b)
                val_losses.append(jscl_loss(z, Y_b, cfg["pretrain_temperature"]).item())
        tr_mean = float(np.mean(tr_losses)) if tr_losses else float("nan")
        val_mean = float(np.mean(val_losses)) if val_losses else float("nan")
        if val_mean < best_val_loss:
            best_val_loss = val_mean
            best_state = {k: v.clone() for k, v in model.adapter.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
        if verbose:
            print(f"  [JSCL] Ep {epoch:3d} | train {tr_mean:.4f} | val {val_mean:.4f} | "
                  f"pat {patience_ctr}/{cfg['pretrain_patience']}")
        if patience_ctr >= cfg["pretrain_patience"]:
            if verbose: print(f"  Early stop at epoch {epoch}.")
            break
    if best_state is not None:
        model.adapter.load_state_dict(best_state)
    for p in model.proto_layer.parameters(): p.requires_grad_(True)
    for p in model.classifier.parameters(): p.requires_grad_(True)
    if verbose: print(f"  Best JSCL val loss: {best_val_loss:.4f}")
    return best_val_loss


def run_one_epoch(model, loader, optimizer, C_jac, class_weights, cfg, device, train=True):
    model.train(train)
    totals = {k: torch.zeros((), device=device) for k in
             ["total", "bce", "clst", "sep", "div", "cntrst", "covar"]}
    n_batches = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for X_b, Y_b in loader:
            X_b = X_b.to(device, non_blocking=True); Y_b = Y_b.to(device, non_blocking=True)
            logits, sim, z = model(X_b)
            l_bce    = bce_loss(logits, Y_b, class_weights)
            l_clst   = clustering_loss(sim, Y_b, model.proto_layer.assignment)
            l_sep    = separation_loss(sim, Y_b, model.proto_layer.assignment)
            l_div    = orthogonality_loss(model.proto_layer.prototypes)
            l_cntrst = contrastive_loss(model.proto_layer.prototypes, C_jac)
            l_covar  = covariance_loss(z)
            l_total = (l_bce + cfg["lambda_clst"]*l_clst + cfg["lambda_sep"]*l_sep
                       + cfg["lambda_div"]*l_div + cfg["lambda_cntrst"]*l_cntrst
                       + cfg.get("lambda_covar", 0.0)*l_covar)
            if train:
                optimizer.zero_grad(set_to_none=True)
                l_total.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            totals["total"] += l_total.detach(); totals["bce"] += l_bce.detach()
            totals["clst"] += l_clst.detach(); totals["sep"] += l_sep.detach()
            totals["div"] += l_div.detach(); totals["cntrst"] += l_cntrst.detach()
            totals["covar"] += l_covar.detach()
            n_batches += 1
    return {k: (v / max(n_batches, 1)).item() for k, v in totals.items()}


@torch.no_grad()
def project_prototypes(model, X_train, Y_train, pid_train, device, verbose=True):
    """Vectorized per-class projection -- one matmul per class, stays
    on `device` throughout (see project efficiency notes elsewhere)."""
    model.eval()
    all_z = []
    for start in range(0, len(X_train), 512):
        xb = torch.tensor(X_train[start:start+512], dtype=torch.float32).to(device)
        all_z.append(model.adapter(xb))
    all_z = torch.cat(all_z, dim=0)
    z_norm = F.normalize(all_z, dim=1)
    p_norm_all = F.normalize(model.proto_layer.prototypes, dim=1)
    scale = model.proto_layer.scale
    proj = {}
    for name in CLASS_NAMES:
        s, e = model.proto_layer.offsets[name]
        class_idx = CLASS_NAMES.index(name)
        class_mask = torch.tensor(Y_train[:, class_idx] == 1, device=device)
        if not class_mask.any():
            continue
        eligible_z = z_norm[class_mask]
        eligible_idxs = torch.where(class_mask)[0]
        p_block = p_norm_all[s:e]
        sims_all = scale * (p_block @ eligible_z.T)
        best_local = sims_all.argmax(dim=1)
        best_train = eligible_idxs[best_local]
        best_sim = sims_all[torch.arange(len(best_local), device=device), best_local]
        for k in range(e - s):
            p_idx = s + k
            train_idx = int(best_train[k].item())
            model.proto_layer.prototypes[p_idx].copy_(all_z[train_idx])
            proj[p_idx] = {"train_idx": train_idx, "patient_id": str(pid_train[train_idx]),
                           "class_idx": class_idx, "class_name": name,
                           "sim_score": float(best_sim[k].item())}
    if verbose:
        print(f"\nProjection complete ({len(proj)} prototypes):")
        for p_idx, info in proj.items():
            print(f"  Proto {p_idx:2d} ({info['class_name']:10s}): "
                  f"patient {info['patient_id']}, sim={info['sim_score']:.4f}")
    return proj


def refit_classifier(model, train_loader, X_val, Y_val, pid_val, class_weights, device,
                     k=5, lr=1e-3, epochs=50, patience=10, verbose=True):
    for p in model.adapter.parameters(): p.requires_grad_(False)
    model.proto_layer.prototypes.requires_grad_(False)
    opt = torch.optim.Adam(model.classifier.parameters(), lr=lr, weight_decay=1e-4)
    best_primary, best_state, counter = -1., None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for X_b, Y_b in train_loader:
            X_b = X_b.to(device, non_blocking=True); Y_b = Y_b.to(device, non_blocking=True)
            logits, _, _ = model(X_b)
            loss = bce_loss(logits, Y_b, class_weights)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        va = evaluate_patient_level(model, X_val, Y_val, pid_val, device, k=k)
        primary = va["_primary"]
        improved = (not np.isnan(primary)) and (primary > best_primary)
        if improved or best_state is None:
            if improved:
                best_primary = primary; counter = 0
            else:
                counter += 1
            best_state = {k_: v.clone() for k_, v in model.classifier.state_dict().items()}
        else:
            counter += 1
        if verbose and epoch % 10 == 0:
            print(f"  Refit Ep {epoch:3d} | patient-level AUPRC outcomes {primary:.4f}")
        if counter >= patience:
            break
    model.classifier.load_state_dict(best_state)
    for p in model.adapter.parameters(): p.requires_grad_(True)
    model.proto_layer.prototypes.requires_grad_(True)
    return best_primary


def train_protoehr(cfg, data, device, verbose=True):
    device = torch.device(device) if isinstance(device, str) else device
    os.makedirs(cfg["output_dir"], exist_ok=True)
    torch.manual_seed(cfg["random_seed"]); np.random.seed(cfg["random_seed"])

    X_train, Y_train, pid_train = data["X_train"], data["Y_train"], data["pid_train"]
    X_val, Y_val, pid_val = data["X_val"], data["Y_val"], data["pid_val"]
    class_weights = data["class_weights"].to(device)
    k = cfg.get("topk_pool_k", 5)

    train_loader = to_loader(X_train, Y_train, cfg["batch_size"], True, device, drop_last=True)

    scale = cfg["adapter_out"] ** 0.5
    model = ProtoEHRNet(cfg, scale, verbose=verbose).to(device)
    C_jaccard = build_jaccard_matrix(Y_train, cfg["prototypes_per_class"], cfg["zero_diagonal_blocks"]).to(device)

    history = []
    if cfg.get("use_jscl_pretrain", False):
        pretrain_adapter_jscl(model, X_train, data["Y_subtype_train"], X_val, data["Y_subtype_val"],
                              cfg, device, verbose=verbose)
        if verbose:
            diag = check_dimension_collapse(model, X_train, Y_train, device)
            print(f"  [dim check, post-Stage-0] dims_90={diag['dims_90']} dims_95={diag['dims_95']}")
            history.append({"iteration": 0, "stage": 0, "epoch": 0, "dims_90": diag["dims_90"], "dims_95": diag["dims_95"]})

    best_iter_metric = -1.0; iter_patience_ctr = 0
    projection_map = {}; current_lr = cfg["lr"]
    overall_best_ckpt = os.path.join(cfg["output_dir"], f"{cfg['run_id']}_stage1_best.pt")
    t_start = time.time()
    dims_check_interval = cfg.get("dims_check_interval", 5)

    for iteration in range(1, cfg["max_iterations"] + 1):
        optimizer = torch.optim.Adam(model.parameters(), lr=current_lr, weight_decay=cfg["weight_decay"])
        if verbose:
            print(f"\nIteration {iteration}/{cfg['max_iterations']} — Stage 1 (lr={current_lr:.2e})")

        best_epoch_metric = -1.0; epoch_patience = 0
        iter_ckpt = os.path.join(cfg["output_dir"], f"{cfg['run_id']}_iter{iteration}_best.pt")

        for epoch in range(1, cfg["max_epochs"] + 1):
            tr = run_one_epoch(model, train_loader, optimizer, C_jaccard, class_weights, cfg, device, train=True)
            va = evaluate_patient_level(model, X_val, Y_val, pid_val, device, k=k)
            primary = va["_primary"]

            epoch_record = {"iteration": iteration, "stage": 1, "epoch": epoch,
                            **{f"train_{k_}": round(v, 5) for k_, v in tr.items()},
                            **{k_: round(v, 5) for k_, v in va.items() if not k_.startswith("_")}}

            if verbose and dims_check_interval and (epoch == 1 or epoch % dims_check_interval == 0):
                diag_ep = check_dimension_collapse(model, X_train, Y_train, device)
                print(f"    [dim check] It{iteration} Ep{epoch:03d} | dims_90={diag_ep['dims_90']} dims_95={diag_ep['dims_95']}")
                epoch_record["dims_90"] = diag_ep["dims_90"]; epoch_record["dims_95"] = diag_ep["dims_95"]

            history.append(epoch_record)

            improved = (not np.isnan(primary)) and (primary > best_epoch_metric)
            if improved or epoch == 1:
                if improved:
                    best_epoch_metric = primary; epoch_patience = 0
                torch.save({"model_state": model.state_dict(), "cfg": cfg}, iter_ckpt)
                if not improved:
                    epoch_patience += 1
            else:
                epoch_patience += 1

            if verbose:
                print(f"  It{iteration} Ep{epoch:03d} | patient-level AUPRC {primary:.4f} | pat {epoch_patience}/{cfg['patience']}")
            if epoch_patience >= cfg["patience"]:
                break

        ckpt = torch.load(iter_ckpt, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state"])

        if best_epoch_metric > best_iter_metric:
            best_iter_metric = best_epoch_metric; iter_patience_ctr = 0
            torch.save(ckpt, overall_best_ckpt)
        else:
            iter_patience_ctr += 1

        projection_map = project_prototypes(model, X_train, Y_train, pid_train, device, verbose=verbose)
        history.append({"iteration": iteration, "stage": 2, "epoch": 0, "auprc_macro_outcomes_only": round(best_epoch_metric, 5)})

        if iter_patience_ctr >= cfg["iter_patience"]:
            ckpt = torch.load(overall_best_ckpt, map_location=device, weights_only=False)
            model.load_state_dict(ckpt["model_state"])
            projection_map = project_prototypes(model, X_train, Y_train, pid_train, device, verbose=verbose)
            break
        current_lr *= cfg["lr_decay"]

    refit_primary = refit_classifier(model, train_loader, X_val, Y_val, pid_val, class_weights, device,
                                     k=k, verbose=verbose)
    history.append({"iteration": "final", "stage": 3, "epoch": 0, "auprc_macro_outcomes_only": round(float(refit_primary), 5)})

    final_metrics = evaluate_patient_level(model, X_val, Y_val, pid_val, device, k=k)
    proto_diag = prototype_diagnostics(model, projection_map)

    if verbose:
        diag_final = check_dimension_collapse(model, X_train, Y_train, device)
        print(f"  [dim check, post-all-training] dims_90={diag_final['dims_90']} dims_95={diag_final['dims_95']}")
        history.append({"iteration": "final", "stage": "dim_check", "epoch": 0, "dims_90": diag_final["dims_90"], "dims_95": diag_final["dims_95"]})
        print(f"\nFinal patient-level AUPRC outcomes-only: {final_metrics['_primary']:.4f}")
        print(f"Prototype concentration: {proto_diag['proto_concentration']:.4f}")
        print(f"Prototype fidelity mean/min: {proto_diag['proto_fidelity_mean']:.4f} / {proto_diag['proto_fidelity_min']:.4f}")
        print(f"Total time: {time.time()-t_start:.0f}s")

    return model, final_metrics, proto_diag, projection_map, history


def print_metrics(results, prevalence, class_names=CLASS_NAMES):
    print(f"\n  {'Metric':<32} Value   Baseline")
    print(f"  {'-'*32}  -----   --------")
    for k in ([f"auroc_{n}" for n in class_names] + ["auroc_macro", "auroc_weighted", "auroc_micro", "auroc_macro_outcomes_only"]):
        marker = " ← AUROC primary" if k == "auroc_macro_outcomes_only" else ""
        print(f"  {k:<32} {results.get(k, float('nan')):.4f}   0.500{marker}")
    print()
    for c, name in enumerate(class_names):
        kk = f"auprc_{name}"
        print(f"  {kk:<32} {results.get(kk, float('nan')):.4f}   {prevalence[c]:.3f}")
    for k in ["auprc_macro", "auprc_macro_outcomes_only"]:
        marker = " ← primary metric" if k == "auprc_macro_outcomes_only" else ""
        print(f"  {k:<32} {results.get(k, float('nan')):.4f}  {marker}")
    print()
    for prefix, better in [("f1", "higher"), ("brier", "lower"), ("bacc", "higher")]:
        for name in class_names:
            kk = f"{prefix}_{name}"
            print(f"  {kk:<32} {results.get(kk, float('nan')):.4f}   ({better} is better)")
        print()


# ============================================================================
# Section 8: UMAP visualization
# ============================================================================
def plot_umap_with_prototypes(model, X_train, Y_train, projection_map, device,
                              n_neighbors=30, min_dist=0.1, random_state=42,
                              figsize=(9, 8), run_id=""):
    import umap
    import matplotlib.patches as mpatches
    model.eval()
    with torch.no_grad():
        all_z = []
        for start in range(0, len(X_train), 512):
            xb = torch.tensor(X_train[start:start+512], dtype=torch.float32).to(device)
            all_z.append(model.adapter(xb).cpu())
        all_z = torch.cat(all_z, dim=0).numpy()
    reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, random_state=random_state, metric="cosine")
    emb_2d = reducer.fit_transform(all_z)

    n_pos_per_row = Y_train.sum(axis=1)
    plot_label = np.full(len(Y_train), "none", dtype=object)
    for c, name in enumerate(CLASS_NAMES):
        mask = (Y_train[:, c] == 1) & (n_pos_per_row == 1)
        plot_label[mask] = name
    plot_label[n_pos_per_row > 1] = "multi"

    palette = {"control": "#9e9e9e", "pe_ghtn": "#e07b39", "gdm": "#3b7dd8", "multi": "#8e44ad", "none": "#d0d0d0"}
    fig, ax = plt.subplots(figsize=figsize)
    for lbl in ["none", "control", "pe_ghtn", "gdm", "multi"]:
        m = (plot_label == lbl)
        if m.sum() == 0: continue
        ax.scatter(emb_2d[m, 0], emb_2d[m, 1], s=8, alpha=0.35, color=palette[lbl], label=f"{lbl} (n={m.sum()})", linewidths=0)

    star_colors = {"control": "#333333", "pe_ghtn": "#8a3b00", "gdm": "#0d3d78"}
    for p_idx, info in projection_map.items():
        tidx = info["train_idx"]; cname = info["class_name"]
        ax.scatter(emb_2d[tidx, 0], emb_2d[tidx, 1], marker="*", s=320, color=star_colors.get(cname, "black"),
                  edgecolors="white", linewidths=1.2, zorder=5)
        ax.annotate(f"{cname}#{p_idx}", (emb_2d[tidx, 0], emb_2d[tidx, 1]), fontsize=8, fontweight="bold",
                   xytext=(4, 4), textcoords="offset points", zorder=6)

    star_handles = [mpatches.Patch(color=c, label=f"{n} prototype") for n, c in star_colors.items()]
    handles, _ = ax.get_legend_handles_labels()
    ax.legend(handles=handles + star_handles, loc="best", fontsize=8, framealpha=0.9)
    ax.set_title(f"Adapter embedding space (UMAP) — {run_id}\nstars = projected prototypes (window-level grounding)")
    ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
    fig.tight_layout()
    return fig, emb_2d, reducer


# ============================================================================
# Section 9: Run both tables, compare
# ============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

cohort_df, label_cols, subtype_cols = load_cohort_labels()

all_results = {}
all_models = {}
for version_label, table_name in MONTH_TABLES.items():
    cfg = copy.deepcopy(DEFAULT_CFG)
    cfg["run_id"] = f"{version_label}_baseline"
    cfg["output_dir"] = f"./proto_runs/{version_label}"

    data = build_month_data(table_name, cohort_df, label_cols, subtype_cols, cfg)
    model, final_metrics, proto_diag, projection_map, history = train_protoehr(cfg, data, device, verbose=True)

    pd.DataFrame(history).to_csv(os.path.join(cfg["output_dir"], f"{cfg['run_id']}_history.csv"), index=False)

    print(f"\n=== {version_label} — final patient-level metrics (top-{cfg['topk_pool_k']} pooled) ===")
    print_metrics(final_metrics, data["prevalence"])

    fig, _, _ = plot_umap_with_prototypes(model, data["X_train"], data["Y_train"], projection_map, device, run_id=version_label)
    fig.savefig(os.path.join(cfg["output_dir"], f"{cfg['run_id']}_umap.png"), dpi=150)
    plt.show()

    all_results[version_label] = final_metrics
    all_models[version_label] = (model, data, projection_map)

print(f"\n{'='*70}\nCOMPARISON\n{'='*70}")
print(f"{'Metric':<30} {'cumulative':>12}")
compare_keys = ["auprc_macro_outcomes_only", "auroc_macro_outcomes_only", "auprc_macro", "auroc_macro"]
for k in compare_keys:
    cv = all_results["cumulative"][k]
    marker = " ← primary" if k == "auprc_macro_outcomes_only" else ""
    print(f"{k:<30} {cv:>12.4f}")

# primary = "auprc_macro_outcomes_only"
# # winner = "cumulative" if all_results["cumulative"][primary]
# print(f"\nHigher patient-level validation {primary}: {winner} "
#       f"({all_results[winner][primary]:.4f}")
#       print(f"\nHigher patient-level validation {primary}: {winner} ({all_results[winner][primary]:.4f} / {all_results['delta' if winner=='cumulative' else 'cumulative'][primary]:.4f})")